In [1]:
import torch
import pandas as pd
from pathlib import Path
from bertopic import BERTopic
from plotly.graph_objs.layout.title import subtitle
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
import spacy
from umap import UMAP
from hdbscan import HDBSCAN

print(f"CUDA available: {torch.cuda.is_available()}")

CUDA available: True


In [2]:
reviews = pd.read_csv(
    Path("data") / "reviews_data.csv",
    dtype={
        "appId": str,
        "language": str,
        "reviewId": str,
        "score": "int8",
        "content": str,
        "thumbsUpCount": "int32",
        "reviewCreatedVersion": str,
        "replyContent": str,
        "appVersion": str,
        "lemmas": str,
    },
    parse_dates=["at", "repliedAt"],
)

In [3]:
map_apps = [
    "pl.mapa_turystyczna.app",
    "de.komoot.android",
    "cz.seznam.mapy",
    "pl.traseo2",
    "menion.android.locus",
    "com.alltrails.alltrails",
]

reviews_outdoor = reviews[(reviews["appId"].isin(map_apps) & (reviews["score"] <= 3))]

texts = reviews_outdoor["content"].dropna().tolist()

In [4]:
reviews_outdoor.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2968 entries, 2 to 65220
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   reviewId              2968 non-null   object        
 1   content               2968 non-null   object        
 2   score                 2968 non-null   int8          
 3   thumbsUpCount         2968 non-null   int32         
 4   reviewCreatedVersion  2445 non-null   object        
 5   at                    2968 non-null   datetime64[ns]
 6   replyContent          2475 non-null   object        
 7   repliedAt             2475 non-null   datetime64[ns]
 8   appVersion            2445 non-null   object        
 9   appId                 2968 non-null   object        
 10  language              2968 non-null   object        
dtypes: datetime64[ns](2), int32(1), int8(1), object(7)
memory usage: 246.4+ KB


In [5]:
reviews_outdoor.head(5)

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,appId,language
2,8cc6a136-dd46-421f-a051-87a755a7e3d0,"Powiem tak , opcje subskrypcji sa coraz częsts...",1,0,NaN,2026-04-16 13:06:15,NaN,NaT,NaN,pl.mapa_turystyczna.app,pl
16,453b7a7a-494b-40d9-b9b6-2684f9b047f2,Wielki MINUS za konieczność ręcznego odznaczen...,1,0,1.15.11,2026-04-06 14:50:07,Dziękujemy za zgłoszenie. Wystarczy na stronie...,2026-04-06 20:33:34,1.15.11,pl.mapa_turystyczna.app,pl
46,1c4bb57d-bb15-4efb-9807-f09b6529f195,ok spoko,3,0,1.15.4,2026-03-08 17:52:15,NaN,NaT,1.15.4,pl.mapa_turystyczna.app,pl
70,57b1d2c5-13ed-4691-986d-b19d381d4eda,"kompletnie bezużyteczne, nie wyznacza trasy",1,0,1.15.6,2026-02-14 10:11:18,Dziękujemy za zgłoszenie. Proszę napisać dokła...,2026-02-14 10:50:35,1.15.6,pl.mapa_turystyczna.app,pl
92,71e67568-9926-4846-b710-d2b3a31da573,Brak możliwości importu plików GPX bezpośredni...,2,0,1.15.4,2026-01-12 18:18:30,Czy chodzi o import nagranych tras na innych u...,2026-01-12 17:32:30,1.15.4,pl.mapa_turystyczna.app,pl


In [6]:
app_names = {
    "pl.mapa_turystyczna.app": "Mapa Turystyczna",
    "de.komoot.android": "Komoot",
    "com.alltrails.alltrails": "AllTrails",
    "menion.android.locus": "Locus Map",
    "cz.seznam.mapy": "Mapy.cz",
    "pl.traseo2": "Traseo",
}

reviews_outdoor["appId"] = reviews_outdoor["appId"].replace(app_names)

In [7]:
nlp = spacy.load("pl_core_news_lg")
polish_stopwords = list(nlp.Defaults.stop_words)

custom_stopwords = [
    "nie",
    "się",
    "na",
    "to",
    "jest",
    "do",
    "po",
    "ale",
    "że",
    "aplikacja",
    "apka",
    "za",
    "bardzo",
    "jak",
    "od",
    "tej",
    "tylko",
    "tej",
    "działa",
    "mapa",
    "mapy",
    "spoko",
    "ok",
    "okej",
    "super",
    "fajnie",
    "fajna",
    "fajny",
    "dobra",
    "dobre",
    "dobry",
    "nawet",
    "trochę",
    "teraz",
    "wcześniej",
    "oki",
    "okok",
]

all_stopwords = polish_stopwords + custom_stopwords

In [8]:
vectorizer_model = CountVectorizer(stop_words=all_stopwords, ngram_range=(1, 2))
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

umap_model = UMAP(
    n_neighbors=10, n_components=5, min_dist=0.0, metric="cosine", random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=15,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    language="multilingual",
    min_topic_size=15,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
)
topics, probs = topic_model.fit_transform(texts)

topic_model.save(
    "models/bertopic_only_maps", serialization="safetensors", save_embedding_model=False
)

# topic_model = BERTopic.load("models/bertopic_only_maps", embedding_model=embedding_model)
# topic_model.update_topics(texts)

2026-04-23 08:06:12,064 - BERTopic - WARNING: You are saving a BERTopic model without explicitly defining an embedding model.If you are using a sentence-transformers model or a HuggingFace model supportedby sentence-transformers, please save the model by using a pointer towards that model.For example, `save_embedding_model='sentence-transformers/all-mpnet-base-v2'`


In [10]:
topics_data = topic_model.get_topic_info()
topics_data.to_csv(Path("visualizations") / "topics_data.csv", index=False)
topics_data

,Topic,Count,Name,Representation,Representative_Docs
0,-1,854,-1_trasy_aplikacji_brak_tras,"[trasy, aplikacji, brak, tras, map, niestety, ...","[Byłoby fajnie, ale ... Miałam zaplanowanych (..."
1,0,310,0_premium_płatna_subskrypcji_zgody,"[premium, płatna, subskrypcji, zgody, aplikacj...",[Completely worthless app. They charge 65euro ...
2,1,245,1_offline_map_premium_mapę,"[offline, map, premium, mapę, map offline, int...","[What for are offline maps, if now I can't pla..."
3,2,158,2_polskiego_języka_języka polskiego_brak języka,"[polskiego, języka, języka polskiego, brak jęz...","[Brak języka polskiego., Brak języka polskiego..."
4,3,137,3_konta_zalogować_konto_sms,"[konta, zalogować, konto, sms, założyć, mogę, ...",[Jak na razie kiepsko. Przy zakładaniu nowego ...
5,4,108,4_map_pobrać_stare_pobrać map,"[map, pobrać, stare, pobrać map, offline, off,...",[Polecam autorom zajrzeć do (nawet starych) wo...
6,5,104,5_trasy_tras_drogi_trasa,"[trasy, tras, drogi, trasa, trasę, polecam, po...","[Bardzo niefajne, że szlaki wiodą przez drogi ..."
7,6,100,6_wiesza_wymaga dopracowania_dopracowania_pote...,"[wiesza, wymaga dopracowania, dopracowania, po...","[Słabizna. Wiesza się, gaśnie. Ok napisze w wo..."
8,7,94,7_rowerowa_rowerowe_rower_trasy,"[rowerowa, rowerowe, rower, trasy, trasa, prow...","[Aplikacja super,ale ma kilka tematów do popra..."
9,8,64,8_wyłącza_aktualizacji_zawiesza_przestała,"[wyłącza, aktualizacji, zawiesza, przestała, o...","[Byłam wielką fanką tej apki, zaplanowałam mnó..."


In [11]:
custom_topic_names = {
    -1: "Szum informacyjny / Różne",
    0: "Wymuszona subskrypcja (Premium)",
    1: "Oszukany tryb offline (Wymaga sieci)",
    2: "Brak języka polskiego",
    3: "Błąd logowania (Weryfikacja SMS/Email)",
    4: "Błąd pobierania / Ogromne rozmiary map",
    5: "Złe wyznaczanie tras (Prowadzi asfaltem/brodem)",
    6: "Aplikacja zawiesza się (Niestabilność)",
    7: "Nieaktualne ścieżki rowerowe",
    8: "Crashe i psucie aplikacji po aktualizacji",
    9: "Brak zapisu tras i utrata zdjęć",
    10: "Złe zliczanie dystansu (Ścinanie zakrętów)",
    11: "Brak nawigacji punkt-do-punktu w terenie",
    12: "Częste gubienie sygnału GPS",
    13: "Brak wsparcia dla Galaxy Watch / Smartwatchy",
    14: "Paywall (Zbyt duża blokada funkcji)",
    15: "Ogólna słaba użyteczność",
    16: "Brak obracania mapy do kierunku ruchu",
    17: "Błędne i zmieniane przez AI nazwy punktów POI",
    18: "Zła nawigacja miejska (Mylenie adresów)",
    19: "Brak lub trudny import plików GPX",
    20: "Nieintuicyjny interfejs (Zły UX)",
    21: "Aplikacja miesza funkcje i zacina się",
    22: "Braki na mapach (Szczyty, ruiny, szlaki)",
    23: "Ekstremalne zużycie baterii",
    24: "Brak kompatybilności (Wybrane smartfony)",
    25: "Zły podział map regionalnych",
    26: "Zamrażanie przy planowaniu długich tras",
    27: "Błędy parowania Bluetooth (Liczniki rowerowe)",
    28: "Blokada sprawdzania subskrypcji w górach",
    29: "Brak lokalnych szlaków nizinnych",
    30: "Nieczytelne, nakładające się ikony tras",
    31: "Brak dostępu do plików systemowych (Android 12/14)",
    32: "Opinie jednosłowne (Słaba)",
    33: "Zbyt dużo inwazyjnych reklam",
    34: "Brak wsparcia dla urządzeń Garmin",
    35: "Opinie jednosłowne (Beznadziejna)",
    36: "Drastyczny spadek płynności / Lagi",
    37: "Nieintuicyjne zaczynanie i usuwanie tras",
}


topic_model.set_topic_labels(custom_topic_names)


topic_model.get_topic_info()

,Topic,Count,Name,CustomName,Representation,Representative_Docs
0,-1,854,-1_trasy_aplikacji_brak_tras,Szum informacyjny / Różne,"[trasy, aplikacji, brak, tras, map, niestety, ...","[Byłoby fajnie, ale ... Miałam zaplanowanych (..."
1,0,310,0_premium_płatna_subskrypcji_zgody,Wymuszona subskrypcja (Premium),"[premium, płatna, subskrypcji, zgody, aplikacj...",[Completely worthless app. They charge 65euro ...
2,1,245,1_offline_map_premium_mapę,Oszukany tryb offline (Wymaga sieci),"[offline, map, premium, mapę, map offline, int...","[What for are offline maps, if now I can't pla..."
3,2,158,2_polskiego_języka_języka polskiego_brak języka,Brak języka polskiego,"[polskiego, języka, języka polskiego, brak jęz...","[Brak języka polskiego., Brak języka polskiego..."
4,3,137,3_konta_zalogować_konto_sms,Błąd logowania (Weryfikacja SMS/Email),"[konta, zalogować, konto, sms, założyć, mogę, ...",[Jak na razie kiepsko. Przy zakładaniu nowego ...
5,4,108,4_map_pobrać_stare_pobrać map,Błąd pobierania / Ogromne rozmiary map,"[map, pobrać, stare, pobrać map, offline, off,...",[Polecam autorom zajrzeć do (nawet starych) wo...
6,5,104,5_trasy_tras_drogi_trasa,Złe wyznaczanie tras (Prowadzi asfaltem/brodem),"[trasy, tras, drogi, trasa, trasę, polecam, po...","[Bardzo niefajne, że szlaki wiodą przez drogi ..."
7,6,100,6_wiesza_wymaga dopracowania_dopracowania_pote...,Aplikacja zawiesza się (Niestabilność),"[wiesza, wymaga dopracowania, dopracowania, po...","[Słabizna. Wiesza się, gaśnie. Ok napisze w wo..."
8,7,94,7_rowerowa_rowerowe_rower_trasy,Nieaktualne ścieżki rowerowe,"[rowerowa, rowerowe, rower, trasy, trasa, prow...","[Aplikacja super,ale ma kilka tematów do popra..."
9,8,64,8_wyłącza_aktualizacji_zawiesza_przestała,Crashe i psucie aplikacji po aktualizacji,"[wyłącza, aktualizacji, zawiesza, przestała, o...","[Byłam wielką fanką tej apki, zaplanowałam mnó..."


In [12]:
distance_map = topic_model.visualize_topics(custom_labels=True)
distance_map.write_html(Path("visualizations") / "intertopic_distance_map.html")

distance_map

In [13]:
visualized_docs = topic_model.visualize_documents(
    texts,
    hide_annotations=False,
    hide_document_hover=False,
    topics=list(range(15)),
    title="Najczęstsze tematy negatywnych recenzji aplikacji outdoorowych",
    custom_labels=True,
)

visualized_docs.write_html(Path("visualizations") / "interactive_problems_map.html")
visualized_docs

In [14]:
classes = reviews_outdoor["appId"].tolist()

topics_per_class = topic_model.topics_per_class(texts, classes)

In [15]:
visualized_classes = topic_model.visualize_topics_per_class(
    topics_per_class,
    top_n_topics=15,
    normalize_frequency=True,
    custom_labels=True,
)

visualized_classes.write_html(Path("visualizations") / "competitor_matrix.html")
visualized_classes

In [16]:
visualized_hierarchy = topic_model.visualize_hierarchy(custom_labels=True)

visualized_hierarchy.write_html(Path("visualizations") / "hierarchy.html")
visualized_hierarchy

In [17]:
topic_model.get_representative_docs(29)

['Kupiłem mapę "Bliskie okolice Wroclawia. Część południowo-wschodnia. Nie dość, że jest to dość kiepskiej jakości skan (kiepska rozdzielczość), to jeszcze nie zawiera okolicznych szlaków rowerowych o których mowa na oficjalnej stronie gminy Długołęka. No i oczywiście zakupiona mapa w żaden sposób nie jest interaktywna - równie dobrze mógłbym jeździć z papierową. Reasumując nie warto było płacić 12zł za kawałeczek nieużyteczne mapki.',
 'Mapa pokazuje szlaki turystyczne TYLKO na najbardziej przełażonych terenach kraju. Na pozostałej części Polski porażka, szlaków turystycznych nie ma... Przykład: Drawienski Park Narodowy - ZERO szlaków na mapie, Spalski Park Krajobrazy- zero szlaków, Bolimowksi- 1/4 szlaków....moge tak wymieniać bez końca. W tej postaci aplikacja jest bardzo niedopracowana, i żartem jest pobieranie za nia oplat w wersji premium. Odinstalowuje i wracam do sprawdzone LOCUSa...',
 'You can\'t just show all the trails. On Mapa Turystyczna it just works, as a default. It\'s

In [18]:
topic_model.find_topics("parking bus logistyka dojazd")

([11, 5, 14, 7, 18],
 [np.float32(0.53408754),
  np.float32(0.4977353),
  np.float32(0.48297602),
  np.float32(0.47779903),
  np.float32(0.4695186)])

In [19]:
topic_model.get_representative_docs(32)

['Słaba', 'Słaba', 'Taka słaba']